# Nutrition Recommender Development 
## Excluding EDA

In [ ]:
#Importing essential libraries
from openpyxl import load_workbook
from faker import Faker
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import string
import random
import csv

#Libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler


## Loading PCOS Dataset 

In [ ]:
#Loading the excel file
pcos_wb = load_workbook("PCOS Dataset.xlsx")

#Selecting the active sheet 
pcos_ws = pcos_wb.active

#Taking all rows as a list of list/tuples 
pcos_data = list(pcos_ws.iter_rows(values_only=True))

#Seperating the header from the actual data rows
headers = pcos_data[0]
data_rows = pcos_data[1:]

#Creating a pandas dataframe
pcos_df = pd.DataFrame(data_rows, columns=headers)

#Priting the table
print(pcos_df.to_string())

#Reading data from the cells
#for row in pcos_ws.iter_rows(values_only=True):
#    print(row)

## Viewing List of Feature Names

In [ ]:
#Viewing all feature names

#Loading the excel file
df1 = pd.read_excel("PCOS Dataset.xlsx")

#Extracting the column names
feature_names = df1.columns.tolist()

#Printing the feature names
print(f"Total features: {len(feature_names)}\n")
for feature in feature_names:
    print(f"'{feature}',")

## Creating a reduced copy of the PCOS dataframe with only the essential features

In [ ]:
#Reduced Dataset

essential = [
    'Age',
    'Height_cm',
    'Weight_kg',
    'BMI',
    'Menstrual_Cycle_Length_days',
    'Menstrual_Irregularity',
    'Fasting_Glucose_mg_dL',
    'Fasting_Insulin_uIU_mL',
    'HOMA_IR',
    'LH_mIU_mL',
    'FSH_mIU_mL',
    'LH_FSH_Ratio',
    'Total_Testosterone_ng_dL',
    'Free_Testosterone_pg_mL',
    'Total_Cholesterol_mg_dL',
    'Triglycerides_mg_dL',
    'Dietary_Sugar_Intake',
    'Physical_Activity_Level',
    'PCOS_Diagnosis', 
    'Hirsutism_Score_FG',
    'Acne_Severity',
    'Alopecia',
    'Skin_Darkening_Acanthosis',
    'Vitamin_D_ng_mL'
]

df2 = df1[essential].copy()

df2.head()

### Viewing description of the features

In [ ]:
df2.describe()

### Median Imputation of Negative Values within Features: Fasting_Insulin_uIU_mL, HOMA_IR, LH_mIU_mL, Free_Testosterone_pg_mL, Triglycerides_mg_dL

In [ ]:
#Median Imputation
#First change negative values to NaN 

cols = [
    'Fasting_Insulin_uIU_mL',
    'HOMA_IR',
    'LH_mIU_mL',
    'Free_Testosterone_pg_mL',
    'Triglycerides_mg_dL',
    'Vitamin_D_ng_mL'
]

#Converting negative values to nan
df2[cols] = df2[cols].mask(df2[cols] < 0, np.nan)

#Checking to see if negative values were removed
print("Remaining negative values: ")
print((df2[cols] < 0).sum())


In [ ]:
#Converting NaN values to median values
for col in ['Fasting_Insulin_uIU_mL', 'HOMA_IR', 'LH_mIU_mL', 'Free_Testosterone_pg_mL', 'Triglycerides_mg_dL', 'Vitamin_D_ng_mL']:
    col_median = df2[col].median()
    df2[col] = df2[col].fillna(col_median)

#Checking that the dataset still contains 468 rows
print(f"Data shape: {df2.shape}")

### Recalculating Ratios 

In [ ]:
#Recalculating LH to FSH Ratio column and HOMA IR column
#For mathematical integrity

#LH/FSH Ratio Formula
df2['LH_FSH_Ratio'] = df2['LH_mIU_mL'] / df2['FSH_mIU_mL']

#HOMA IR Formula
df2['HOMA_IR'] = (df2['Fasting_Insulin_uIU_mL'] * df2['Fasting_Glucose_mg_dL']) / 405

In [ ]:
#Replacing zero values with NaN
for col in ['Fasting_Insulin_uIU_mL', 'Triglycerides_mg_dL']:
    df2[col] = df2[col].replace(0.0, np.nan)
    col_median = df2[col].median()
    df2[col] = df2[col].fillna(col_median)
    
#Recalculating HOMA IR again
#HOMA IR Formula
df2['HOMA_IR'] = (df2['Fasting_Insulin_uIU_mL'] * df2['Fasting_Glucose_mg_dL']) / 405

#Checking to see if zero values were removed
print("Remaining zero values: ")
print((df2[['Fasting_Insulin_uIU_mL', 'Triglycerides_mg_dL', 'HOMA_IR']] == 0).sum())


### Identifying Outliers

In [ ]:
#
cont_cols = ['Fasting_Glucose_mg_dL', 'Fasting_Insulin_uIU_mL', 'HOMA_IR', 'LH_mIU_mL', 'FSH_mIU_mL', 'LH_FSH_Ratio', 'Total_Testosterone_ng_dL', 'Free_Testosterone_pg_mL', 'Total_Cholesterol_mg_dL', 'Triglycerides_mg_dL', 'Hirsutism_Score_FG', 'Vitamin_D_ng_mL']

outlier_data = []

for col in cont_cols: 
    
    #Calculating the quartiles and IQR
    Q1 = df2[col].quantile(0.25)
    Q3 = df2[col].quantile(0.75)
    IQR = Q3 - Q1

    #Calculating the lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    #Finding the rows that breach the bounds
    outliers = df2[(df2[col] < lower_bound) | (df2[col] > upper_bound)]
    outlier_count = len(outliers)
    percentage = (outlier_count / len(df2)) * 100

    outlier_data.append({
        'Feature': col,
        'Number of Outliers': outlier_count,
        'Percentage of Outliers': percentage,
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2)
    })

#Compiling the outlier data for each feature into a dataframe
df_outliers = pd.DataFrame(outlier_data)

#Sorting the dataframe by order of the number of outliers
df_outliers = df_outliers.sort_values(by='Number of Outliers', ascending=False)

#Viewing the dataframe
df_outliers

In [ ]:
#Saving the cleaned dataset to a new file
df2.to_excel('cleandata.xlsx', index=False)

In [ ]:
##Assigning Fake First Name, Surname, Emails, and Passwords to Data for Testing
fake = Faker()

#Function to create the fake password
def gen_password(length=12):
    chars = string.ascii_letters + string.digits + "!@#$%^&*"
    return ''.join(random.choice(chars) for _ in range(length))

first_names = []
last_names = []
emails = []
passwords = []

#Creating first names, last names and passwords to go with passwords
for i in range (len(df2)):

    #Creating the first name, last name and emails
    firstname = fake.first_name()
    lastname = fake.last_name()

    first_names.append(firstname)
    last_names.append(lastname)
    emails.append(f"{firstname.lower()}.{lastname.lower()}{i}@example.com")
    passwords.append(gen_password())

df2["first_name"] = first_names
df2["last_name"] = last_names
df2["email"] = emails
df2["password"] = passwords


In [ ]:
#Saving the fake credentials to the file
df2.to_excel('cleandata.xlsx', index=False)

# Creating a sythentic target variable: Nutrient vector
### Creating the python logic for the features to decide the level of nutrients needed

### Connecting target nutrition vector levels to suitable foods
### Cosine Similarity

In [ ]:
#Target nutrients matching to nutrients names in the USDA Nutrition/Food datasets

#Loading the nutrition datasets
df_food = pd.read_csv('food.csv')
df_food_nutrient = pd.read_csv('food_nutrient.csv')
df_nutrient = pd.read_csv('nutrient.csv')

#Seperating the target nutrients using the USDA nutrient ids
usda_ids = [291, 646, 304]
df_selected_nutrients = df_nutrient[df_nutrient['nutrient_nbr'].isin(usda_ids)]

#Relational Merge Pipeline
#Linking the filtered nutrients to the bridge table
merge_part1 = pd.merge(df_food_nutrient, df_selected_nutrients, left_on='nutrient_id', right_on='id', how='inner')

#Linking the result to food name table
df_joined = pd.merge(merge_part1, df_food, on='fdc_id', how='inner')

#Converting the table from a vertical format to horizontal format
#Each row represents one unique food
food_matrix = df_joined.pivot_table(
    index='description',
    columns='name',
    values='amount',
    aggfunc='mean'
).fillna(0)

#Vector for the target nutrients 
target_nutrients = [
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg' 
]

filtered_food_matrix = food_matrix[target_nutrients]

#Cosine similarity reccomender function
def recommend_food(patient_vector, food_db, top_n=10):

    #Converting patient vectors to a 2D row array
    vector_array_2d = np.array(patient_vector).reshape(1, -1)

    #Calculating the Cosine Similarity across all matrix rows simultaneously 
    similar_scores = cosine_similarity(vector_array_2d, food_db)[0]

    #Compiling the results into a readable output table
    results_df = food_db.copy()
    results_df['Match Score (%)'] = np.round(similar_scores * 100, 2)

    #Sorting from highest geometric match to lowest
    return results_df.sort_values(by='Match Score (%)', ascending=False).head(top_n)

patient_vector_test = [35.0, 20.0, 400.0]

top_reccomendations = recommend_food(patient_vector_test, filtered_food_matrix, top_n=5)
print("Top matching food reccomended:")
print(top_reccomendations)


### Fixing Feature Scaling with Cosine Similarity for Food Recommendation 

In [ ]:
# New Cosine Similarity Food recommendation engine

scaler = MinMaxScaler()
food_matrix_scaled = scaler.fit_transform(filtered_food_matrix)
food_scaled = pd.DataFrame(food_matrix_scaled, columns=filtered_food_matrix.columns, index=filtered_food_matrix.index)

def food_recommender_scaled(patient_vector, food_db, food_scaled_db, scalerobj, top_n=10):
    #Applying the MinMax scaling to the patient vector before running Cosine Similarity
    #Converting the patient vector to 2D
    vector_2d = np.array(patient_vector).reshape(1, -1)

    #Scaling the patient vector using the exact same food scaler rules
    #Calculating the similarity using the scaled spaces
    scaled_patient_vector = scalerobj.transform(vector_2d)
    score_similarity = cosine_similarity(scaled_patient_vector, food_scaled_db)[0]

    #Attaching the scores back to the original unscaled food database 
    results_df = food_db.copy()
    results_df['Match Score (%):'] = np.round(score_similarity * 100, 2)

    return results_df.sort_values(by='Match Score (%):', ascending=False).head(top_n)

test_patient = [35.0, 20.0, 400.0]
test_patient2 = [25.0, 12.0, 450.0]

top_reccomendations1 = food_recommender_scaled(test_patient, filtered_food_matrix, food_matrix_scaled, scaler, top_n=10)
print("Top matching food reccomended:")
print(top_reccomendations1)

top_reccomendations2 = food_recommender_scaled(test_patient2, filtered_food_matrix, food_matrix_scaled, scaler, top_n=10)
print("Top matching food reccomended:")
print(top_reccomendations2)

## Creating a new nutrition vector with multipliers for nutrient amounts

In [ ]:
#Nutrient reccomender python logic

def nutrient_vector_2(patient_record):

    # Set a baseline daily reccomended intake of nutrients
    magnesium = 320.0 #milligrams
    fibre = 25.0 #grams
    PUFA = 12.0 #grams
    zinc = 7.0 #milligrams 25mg max
    vitamin_d = 10.0 #micrograms (1000 times smaller than a milligram) max 50ug

    #Fiber reccomendation logic
    #High HOMA IR or high Fasting Glucose level can indicate insulin resistance
    #Using a continuous proportional multiplier with 15g as a safety cap      
    if patient_record['HOMA_IR'] > 1.9 or patient_record['Fasting_Glucose_mg_dL'] > 99:
        fibre_addition = patient_record['HOMA_IR'] - 1.9
        fibre += min(15.0, fibre_addition * 1.5)

    #Omega 3 / Polyunsaturated fat reccommendation logic
    #High triglycerides and the presence of severe acne can indicate high lipids and inflammation
    #Omega 3 can lower lipid levels and combat skin inflammation
    #Using a continuous proportional multiplier with 10g as a safety cap
    if 150 <= patient_record['Triglycerides_mg_dL'] > 199 and patient_record['Acne_Severity'] == 3:
        PUFA_addition = patient_record['Triglycerides_mg_dL'] - 199
        PUFA += min(10.0, PUFA_addition * 1.8)

    if 150 <= patient_record['Triglycerides_mg_dL'] > 199 and patient_record['Acne_Severity'] == 2:
        PUFA_addition = patient_record['Triglycerides_mg_dL'] - 199
        PUFA += min(10.0, PUFA_addition * 1.5)

    if 150 <= patient_record['Triglycerides_mg_dL'] > 199 and patient_record['Acne_Severity'] == 1:
        PUFA_addition = patient_record['Triglycerides_mg_dL'] - 199
        PUFA += min(10.0, PUFA_addition * 1.3)

    #Magnesium reccomendation logic
    #Magnesium can support insulin resistance and hormonal imbalance
    #Using a continuous proportional multiplier with 80mg as a safety cap
    if patient_record['HOMA_IR'] > 1.9 and patient_record['PCOS_Diagnosis'] == 1:
        magnesium_addition = patient_record['HOMA_IR'] - 1.9
        magnesium += min(80, magnesium_addition * 10)
    
    if patient_record['HOMA_IR'] > 1.9 and patient_record['PCOS_Diagnosis'] == 0:
            magnesium_addition_2 = patient_record['HOMA_IR'] - 1.9
            magnesium += min(80, magnesium_addition_2 * 7.5)
    
    #Vitamin D recommendation logic
    #Vitamin D can support patients with hormonal imbalance, insulin resistance and hirsutism
    #Using a continuous proportional multiplier with 80mg as a safety cap
    if patient_record['BMI'] > 25 and patient_record['Vitamin_D_ng_mL'] < 10:
        vitamin_d_addition = patient_record['BMI'] - 25
        vitamin_d += min(40, vitamin_d_addition * 2.5)

    if patient_record['BMI'] > 25 and patient_record['Vitamin_D_ng_mL'] < 20:
            vitamin_d_addition = patient_record['BMI'] - 25
            vitamin_d += min(40, vitamin_d_addition * 1.5)

    #Zinc recommendation logic
    #Zinc can support PCOS patients with hirsutism, alopecia
    if patient_record['Total_Testosterone_ng_dL'] > 46:
        zinc_addition = patient_record['Total_Testosterone_ng_dL'] - 46
        zinc += min(18, zinc_addition * 1.5)

    #Round the nutrient figures
    return [round(fibre, 1), round(PUFA, 1), round(magnesium, 1), round(vitamin_d, 1), round(zinc, 1)]

#Testing the reccomender logic on first patient
patient_1 = df2.iloc[0]
target_vector = nutrient_vector_2(patient_1)

print(f"Target Nutrient Vector for patient 1: {target_vector}")


In [ ]:
#creating a copy of the dataframe
df3 = df2.copy(deep=True)
df3.head()


In [ ]:
#Running the nutrient vector logic across all rows in the PCOS dataset
df3['Target_Nutrient_Vector'] = df3.apply(nutrient_vector_2, axis=1)

print("Dataset with integrated target nutrient vector:")
print(df3[['Fasting_Glucose_mg_dL', 'HOMA_IR', 'Triglycerides_mg_dL', 'Acne_Severity', 'Target_Nutrient_Vector']].head())

In [ ]:
#Save nutrient vector data to a new file
df3.head()
df3.to_csv("patientdata.csv", index=False)

# Creating food matrix
### Creating the food matrix so that foods can be filtered based on the 5 specific target nutrients

In [ ]:
#Target nutrients matching to nutrients names in the USDA Nutrition/Food datasets

#Loading the nutrition datasets
df_food = pd.read_csv('food.csv')
df_food_nutrient = pd.read_csv('food_nutrient.csv')
df_nutrient = pd.read_csv('nutrient.csv')

#Seperating the target nutrients using the USDA nutrient ids
target_usda_ids = [291, 646, 304, 309, 325, 326]
df_selected_nutrients_2 = df_nutrient[df_nutrient['nutrient_nbr'].isin(target_usda_ids)]

#Relational Merge Pipeline
#Linking the filtered nutrients to the bridge table
merge_part_1 = pd.merge(df_food_nutrient, df_selected_nutrients_2, left_on='nutrient_id', right_on='id', how='inner')

#Linking the result to food name table
df_joined_2 = pd.merge(merge_part_1, df_food, on='fdc_id', how='inner')

#Converting the table from a vertical format to horizontal format
#Each row represents one unique food
food_matrix_2 = df_joined_2.pivot_table(
    index='description',
    columns='name',
    values='amount',
    aggfunc='mean'
).fillna(0)

food_matrix_2['Vitamin_D_Total_UG'] = food_matrix_2['Vitamin D2 (ergocalciferol)'] + food_matrix_2['Vitamin D3 (cholecalciferol)']

#Vector for the target nutrients 
nutrients_5d_order = [
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
]

food_matrix_5d = food_matrix_2[nutrients_5d_order]

# New Cosine Similarity Food recommendation engine
scaler_2 = MinMaxScaler()
food_matrix_scaled_2 = scaler_2.fit_transform(food_matrix_5d)
food_scaled_2 = pd.DataFrame(food_matrix_scaled_2, columns=food_matrix_5d.columns, index=food_matrix_5d.index)

test_patient_vector = [np.float64(28.6), 22.0, np.float64(361.4), np.float64(12.0), np.float64(16.2)]

recommendation_test = food_recommender_scaled(test_patient_vector, food_matrix_5d, food_matrix_scaled_2, scaler_2, top_n=10)
print("Top matching food reccomended:")
print(recommendation_test)

In [ ]:
#food_matrix_5d.to_csv("food_matrix_5d.csv", index=False)

### Nutrient Contribution Test
### Calculating how much recommended foods contribute to target nutrient levels

### Seeing how diverse the recommendations is with 5 target nutrients

In [ ]:
#New food catalogue coverage check with new cosine similarity food recommender

def scaled_catalog_coverage_2(patient_df, food_db, food_scaled_db, scalerobj, top_n=10):

    #Defining the set for unique foods recommended
    unique_foods_recommended = set()

    for index, patient_record in patient_df.iterrows():
        patient_vector = nutrient_vector_2(patient_record)

        #Scaled food recommendation function
        top_foods_df = food_recommender_scaled(
            patient_vector, food_db, food_scaled_db, scalerobj, top_n=top_n
        )

        unique_foods_recommended.update(top_foods_df.index.tolist())
    
    #Calculating the coverage percentage
    coverage_percentage = (len(unique_foods_recommended) / len(food_db)) * 100

    #Printing the metrics 
    print("Food Recommender Catalog Coverage")
    print(f"Total Unique Foods in Food Database: {len(food_db)}")
    print(f"Number of Unique Foods Recommended: {len(unique_foods_recommended)}")
    print(f"Percentage of Unique Foods Recommended out of Total Foods: {coverage_percentage:.2f}%")

    return unique_foods_recommended

#Running the catalog coverage function
unique_foods_3 = scaled_catalog_coverage_2(df3, food_matrix_5d, food_matrix_scaled_2, scaler_2, top_n=10)


### Adding Weighted Cosine Similarity into the Food Recommender 
### To test if catalog coverage can be improved 

In [ ]:
##Weighted cosine similarity
#A dynamic clinical weight vector adds a weight to nutrients depending on a patient's specific needs

def food_recommender_weighted(patient_vector, food_db, food_scaled_db, scalerobj, top_n=10):
    #Estabilishing clinical priority weights based on the patient's specific presentation
    #Default weights are equal 
    weights = np.array([1.0, 1.0, 1.0, 1.0, 1.0])

    #If insulin resistance = severe -> prioritise Fiber (index 0) and Magnesium (index 2)
    if patient_vector[0] > 30.0 or patient_vector[2] > 380.0:
        weights[0] = 2.5 #High geometric priority to Fiber
        weights[2] = 2.0 #High geometric priority to Magnesium

    #If hyperandrogenism is severe prioritise Zinc (index 3)
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scaled_patient_vector = scalerobj.transform(vector_2d)

    #Applying the clinical weights
    #Multiplying both the patient vector and database rows by weights
    weighted_patient_vector = scaled_patient_vector * weights
    weighted_food_db = food_scaled_db * weights
    
    #Calculating the similarity and returning the results
    score_similarity = cosine_similarity(weighted_patient_vector, weighted_food_db)[0]

    results_df = food_db.copy()
    results_df['Match_Score (%)'] = np.round(score_similarity * 100, 2)
    return results_df.sort_values(by='Match_Score (%)', ascending=False).head(top_n)

test_patient_vector_2 = [28.6, 22.0, 361.4, 12.1, 16.2]

weighted_test = food_recommender_weighted(test_patient_vector_2, food_matrix_5d, food_matrix_scaled_2, scaler_2, top_n=10)
print("Top matching food reccomended:")
print(weighted_test)

### Diversifying the food categories recommended in the top 10

In [ ]:
#Food category diversification
#Programming the recommender to return the top food matches
#while enforcing a strict limit on how many items can share the same category
food_df = pd.read_csv('food.csv')
category_df = pd.read_csv('food_category.csv')

#Renaming columns in the category description 
#To avoid conflict with the food description
clean_category_df = category_df.rename(columns={'id':'food_category_id', 'description':'category_name'})

#Merging category names onto the food list
df_food_mapped = pd.merge(
    food_df[['description', 'food_category_id']],
    clean_category_df[['food_category_id', 'category_name']],
    on = 'food_category_id',
    how='left'
).drop_duplicates(subset=['description'])

#Attaching the new category name to the food_matrix_5d data
food_matrix_5d_2 = food_matrix_5d.copy()
food_matrix_5d_2 = food_matrix_5d_2.merge(
    df_food_mapped.set_index('description')[['category_name']],
    left_index=True,
    right_index=True,
    how='left'
)

#Filling in missing categories if there are any
food_matrix_5d_2['category_name'] = food_matrix_5d_2['category_name'].fillna('Other')


#Diversified Recommender Logic
def diversified_recommendations(patient_vector, food_df, max_per_category=2):
    #Isolating the nutrients
    nutrients_5d_order = [
        'Fiber, total dietary', 
        'Fatty acids, total polyunsaturated',
        'Magnesium, Mg',
        'Vitamin_D_Total_UG',
        'Zinc, Zn'
    ]
    food_nutrient_matrix = food_df[nutrients_5d_order].values

    #Applying the MinMax scaling to the patient vector before running Cosine Similarity
    #Converting the patient vector to 2D
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scalerobj = MinMaxScaler()
    scaled_food_db = scalerobj.fit_transform(food_nutrient_matrix)

    #Scaling the patient vector using the exact same food scaler rules
    #Calculating the similarity using the scaled spaces
    scaled_patient_vector = scalerobj.transform(vector_2d)
    score_similarity = cosine_similarity(scaled_patient_vector, scaled_food_db)[0]

    #Attaching the scores back to the original unscaled food database 
    results_df = food_df.copy()
    results_df['Match_Score'] = np.round(score_similarity * 100, 2)
    #sorting the results
    sorted_results = results_df.sort_values(by='Match_Score', ascending=False)

    #Diversification loop
    diversified_top_10 = []
    category_counts = {}

    for index, row in sorted_results.iterrows():
        #Getting the category of current food
        current_category = row['category_name']

        #Initialising the category in the tracker
        if current_category not in category_counts:
            category_counts[current_category] = 0

        #If the limit hasn't been reached for the specific category, add the food
        if category_counts[current_category] < max_per_category:
            diversified_top_10.append(row)
            category_counts[current_category] += 1
        
        #Stop the loop once 10 diverse items reached
        if len(diversified_top_10) == 10:
            break

    #Converting the list of rows into a clean Pandas DataFrame
    return pd.DataFrame(diversified_top_10)

test_patient_vector_2 = [28.6, 22.0, 361.4, 12.1, 16.2]

final_recs = diversified_recommendations(test_patient_vector_2, food_matrix_5d_2, max_per_category=2)

print(final_recs[['category_name', 'Match_Score']])



### Checking to see if the coverage changed

In [ ]:
#Calculating coverage with diversified recommender

def scaled_catalog_coverage_3(patient_vector, food_db, top_n=10):

    #Defining the set for unique foods recommended
    unique_foods_recommended = set()
    
    for index, patient_record in patient_vector.iterrows():
        top_foods = diversified_recommendations(test_patient_vector_2, food_matrix_5d_2, max_per_category=2)

    unique_foods_recommended.update(top_foods.index.tolist())

    #Calculating the coverage percentage
    coverage_percentage = (len(unique_foods_recommended) / len(food_db)) * 100

    #Printing the metrics 
    print("Food Recommender Catalog Coverage")
    print(f"Total Unique Foods in Food Database: {len(food_db)}")
    print(f"Number of Unique Foods Recommended: {len(unique_foods_recommended)}")
    print(f"Percentage of Unique Foods Recommended out of Total Foods: {coverage_percentage:.2f}%")

unique_foods_4 = scaled_catalog_coverage_3(df3, food_matrix_5d_2, top_n=10)

### Filtering out food samples and shortening food names for readability

In [ ]:
##Fixing the filtering 
food_data = pd.read_csv('food.csv')

#Filtering for only master food records and filtering out lab tests and sub samples
valid_data_types=['foundation_food', 'sr_legacy_food']
food_data = food_data[food_data['data_type'].isin(valid_data_types)]

#Grouping similar foods together
food_data['short_name'] = food_data['description'].apply(lambda x: ', '.join(str(x).split(',')[:2]))

#Relational Merge Pipeline
#Linking the filtered nutrients to the bridge table
merge_part_1 = pd.merge(df_food_nutrient, df_selected_nutrients_2, left_on='nutrient_id', right_on='id', how='inner')

#Linking the result to food name table
df_joined_3 = pd.merge(merge_part_1, food_data, on='fdc_id', how='inner')

#Using the short name to merge duplicate foods
food_matrix_3 = df_joined_3.pivot_table(
    index='short_name',
    columns='name',
    values='amount',
    aggfunc='mean'
).fillna(0)

food_matrix_3['Vitamin_D_Total_UG'] = food_matrix_3['Vitamin D2 (ergocalciferol)'] + food_matrix_3['Vitamin D3 (cholecalciferol)']

food_matrix_5d_3 = food_matrix_3[nutrients_5d_order]

# New Cosine Similarity Food recommendation engine
scaler_2 = MinMaxScaler()
food_matrix_scaled_3 = scaler_2.fit_transform(food_matrix_5d_3)
food_scaled_3 = pd.DataFrame(food_matrix_scaled_3, columns=food_matrix_5d_3.columns, index=food_matrix_5d_3.index)

test_patient_vector_2 = [28.6, 22.0, 361.4, 12.1, 16.2]

recs = food_recommender_scaled(test_patient_vector_2, food_matrix_5d_3, food_matrix_scaled_3, scaler_2, top_n=10)
print("Top matching food reccomended:")
print(recs)


### Viewing the coverage after filtering

In [ ]:
scaled_catalog_coverage_2(df3, food_matrix_5d_3, food_matrix_scaled_3, scaler_2, top_n=10)

### Introducing Hard Clinical Constraints for elevated target vitamin D and Zinc needs

In [ ]:
#Hard clinical constraints to introduce hybrid filtering 

#Vitamin D is not present in a lot of foods
#Recommended foods show up with near zero Vitamin D due to
#Vector Dot Product
#Before running the Cosine Similarity Calculation:
#Check for a patient's elevated need for Vitamin D
#If they do, the recommender should apply a filter to the food matrix
#To restrict the search space to find items that contain Vitamin D

def food_recommender_scaled_2(patient_vector, food_db, top_n=10):

    #Nutrient order
    #target_fiber = patient_vector[0]
    #target_pufa = patient_vector[1]
    #target_magnesium = patient_vector[2]
    target_vit_d = patient_vector[3]
    target_zinc = patient_vector[4]

    #Creating copies of databases for filtering
    filtered_food_db = food_db.copy()
    
    #Scaled database
    #filtered_scaled_db = food_scaled_db.copy()

    #Elevated need for vitamin D
    if target_vit_d > 25:
        #Filtering out the foods that don't contain a lot of Vitamin D
        vit_d_mask = food_db['Vitamin_D_Total_UG'] > 1.0
        filtered_food_db = filtered_food_db[vit_d_mask]
        #filtered_scaled_db = filtered_scaled_db[filtered_scaled_db.index.isin(filtered_food_db.index)]

    #Elevated need for Zinc
    if target_zinc > 12:
        #Filtering out the foods that don't contain a lot of Zinc
        zinc_mask = food_db['Zinc, Zn'] > 0.5
        filtered_food_db = filtered_food_db[zinc_mask]
        #filtered_scaled_db = filtered_scaled_db[filtered_scaled_db.index.isin(filtered_food_db.index)]

    #If the filtering is too restrictive and returns nothing, reset to the full database
    if filtered_food_db.empty:
        filtered_food_db = food_db.copy()
        #filtered_scaled_db = food_scaled_db.copy()
    
    #Isolating numeric columns from the filtered database
    nutrients_5d_order = [
        'Fiber, total dietary', 
        'Fatty acids, total polyunsaturated',
        'Magnesium, Mg',
        'Vitamin_D_Total_UG',
        'Zinc, Zn'
    ]
    #Filtering
    food_numeric_matrix = filtered_food_db[nutrients_5d_order].values

    #Converting the patient vector to 2D and scaling it normally
    scaler = MinMaxScaler()
    scaled_food_db = scaler.fit_transform(food_numeric_matrix)
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scaled_patient_vector = scaler.transform(vector_2d)

    #Calculating the similarity strictly across the subspace
    score_similarity = cosine_similarity(scaled_patient_vector, scaled_food_db)[0]
    
    #Creating count for the top 10 recommended foods

    #Attaching the scores back and treturning the top results
    results_df = filtered_food_db.copy()
    results_df['Match_Score (%)'] = np.round(score_similarity * 100, 2)
    return results_df.sort_values(by='Match_Score (%)', ascending=False).head(top_n)

test_patient_vector_2 = [28.6, 22.0, 361.4, 12.1, 16.2]
food_recommender_scaled_2(test_patient_vector_2, food_matrix_5d_3, top_n=10)
        

### Evaluating if the hard clinical constraints improved 

In [ ]:
def scaled_catalog_coverage_3(patient_df, food_db, top_n=10):

    #Defining the set for unique foods recommended
    unique_foods_recommended = set()

    for index, patient_record in patient_df.iterrows():
        patient_vector = nutrient_vector_2(patient_record)

        #Scaled food recommendation function
        top_foods_df = food_recommender_scaled_2(patient_vector, food_db, top_n=top_n)

        unique_foods_recommended.update(top_foods_df.index.tolist())
    
    #Calculating the coverage percentage
    coverage_percentage = (len(unique_foods_recommended) / len(food_db)) * 100

    #Printing the metrics 
    print("Food Recommender Catalog Coverage")
    print(f"Total Unique Foods in Food Database: {len(food_db)}")
    print(f"Number of Unique Foods Recommended: {len(unique_foods_recommended)}")
    print(f"Percentage of Unique Foods Recommended out of Total Foods: {coverage_percentage:.2f}%")

    return unique_foods_recommended

scaled_catalog_coverage_3(df3, food_matrix_5d_3, top_n=10)

### Hard Clinical Constraints for Magnesium, PUFAs, and Fiber

In [ ]:
#Hard clinical constraints to introduce hybrid filtering 

#Vitamin D is not present in a lot of foods
#Recommended foods show up with near zero Vitamin D due to
#Vector Dot Product
#Before running the Cosine Similarity Calculation:
#Check for a patient's elevated need for Vitamin D
#If they do, the recommender should apply a filter to the food matrix
#To restrict the search space to find items that contain Vitamin D

def food_recommender_scaled_3(patient_vector, food_db, top_n=10):

    #Nutrient order
    target_fiber = patient_vector[0]
    target_pufa = patient_vector[1]
    target_magnesium = patient_vector[2]
    target_vit_d = patient_vector[3]
    target_zinc = patient_vector[4]

    #Creating copies of databases for filtering
    filtered_food_db = food_db.copy()

    #Elevated need for Magnesium
    if target_fiber > 25:
        #Filtering out the foods that don't contain a lot of Zinc
        fiber_mask = food_db['Fiber, total dietary'] > 0.5
        filtered_food_db = filtered_food_db[fiber_mask]

    #Elevated need for Magnesium
    if target_pufa > 12:
        #Filtering out the foods that don't contain a lot of Zinc
        pufa_mask = food_db['Fatty acids, total polyunsaturated'] > 0.5
        filtered_food_db = filtered_food_db[pufa_mask]

    #Elevated need for Magnesium
    if target_magnesium > 320:
        #Filtering out the foods that don't contain a lot of Zinc
        magnesium_mask = food_db['Magnesium, Mg'] > 0.5
        filtered_food_db = filtered_food_db[magnesium_mask]

    #Elevated need for vitamin D
    if target_vit_d > 10:
        #Filtering out the foods that don't contain a lot of Vitamin D
        vit_d_mask = food_db['Vitamin_D_Total_UG'] > 1.0
        filtered_food_db = filtered_food_db[vit_d_mask]

    #Elevated need for Zinc
    if target_zinc > 7:
        #Filtering out the foods that don't contain a lot of Zinc
        zinc_mask = food_db['Zinc, Zn'] > 0.5
        filtered_food_db = filtered_food_db[zinc_mask]

    #If the filtering is too restrictive and returns nothing, reset to the full database
    if filtered_food_db.empty:
        filtered_food_db = food_db.copy()
    
    #Isolating numeric columns from the filtered database
    nutrients_5d_order = [
        'Fiber, total dietary', 
        'Fatty acids, total polyunsaturated',
        'Magnesium, Mg',
        'Vitamin_D_Total_UG',
        'Zinc, Zn'
    ]
    #Filtering
    food_numeric_matrix = filtered_food_db[nutrients_5d_order].values

    #Converting the patient vector to 2D and scaling it normally
    scaler = MinMaxScaler()
    scaled_food_db = scaler.fit_transform(food_numeric_matrix)
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scaled_patient_vector = scaler.transform(vector_2d)

    #Calculating the similarity strictly across the subspace
    score_similarity = cosine_similarity(scaled_patient_vector, scaled_food_db)[0]
    
    #Creating count for the top 10 recommended foods

    #Attaching the scores back and treturning the top results
    results_df = filtered_food_db.copy()
    results_df['Match_Score (%)'] = np.round(score_similarity * 100, 2)
    return results_df.sort_values(by='Match_Score (%)', ascending=False).head(top_n)

test_patient_vector_2 = [28.6, 22.0, 361.4, 12.1, 16.2]
food_recommender_scaled_3(test_patient_vector_2, food_matrix_5d_3, top_n=10)
        

In [ ]:
def scaled_catalog_coverage_4(patient_df, food_db, top_n=10):

    #Defining the set for unique foods recommended
    unique_foods_recommended = set()

    for index, patient_record in patient_df.iterrows():
        patient_vector = nutrient_vector_2(patient_record)

        #Scaled food recommendation function
        top_foods_df = food_recommender_scaled_3(patient_vector, food_db, top_n=top_n)

        unique_foods_recommended.update(top_foods_df.index.tolist())
    
    #Calculating the coverage percentage
    coverage_percentage = (len(unique_foods_recommended) / len(food_db)) * 100

    #Printing the metrics 
    print("Food Recommender Catalog Coverage")
    print(f"Total Unique Foods in Food Database: {len(food_db)}")
    print(f"Number of Unique Foods Recommended: {len(unique_foods_recommended)}")
    print(f"Percentage of Unique Foods Recommended out of Total Foods: {coverage_percentage:.2f}%")

    return unique_foods_recommended

scaled_catalog_coverage_4(df3, food_matrix_5d_3, top_n=10)